In [81]:
from pathlib import Path

import geopandas as gpd
import networkx as nx
import osmnx as ox
import polars as pl

In [82]:
ROOT_PATH = Path(".").resolve().absolute()
DATASET_PATH = ROOT_PATH / "sumo/simulations/ohare-chicago-junctionless/output/fcd.parquet"
NETWORK_PATH = ROOT_PATH / "networks/graphml/ohare_network.graphml"

In [83]:
G = ox.load_graphml(NETWORK_PATH)
edges_gdf = ox.graph_to_gdfs(G, nodes=False, fill_edge_geometry=True).to_crs(epsg=4326)
edges_gdf


,,,osmid,highway,lanes,name,oneway,ref,reversed,length,geometry,maxspeed,bridge
u,v,key,,,,,,,,,,,
1153867776,29839280,0,985432262,primary,3,Mannheim Road,True,US 12;US 45,False,22.870702,"LINESTRING (-87.87914 41.9887, -87.87925 41.98...",NaN,NaN
29839280,1153867817,0,985432262,primary,3,Mannheim Road,True,US 12;US 45,False,38.594048,"LINESTRING (-87.87925 41.98889, -87.87943 41.9...",NaN,NaN
1153867781,1153867758,0,11537207,motorway_link,1,NaN,True,NaN,False,30.909291,"LINESTRING (-87.88111 41.98211, -87.88126 41.9...",NaN,NaN
1153867758,102856723,0,11537207,motorway_link,1,NaN,True,NaN,False,19.860727,"LINESTRING (-87.88126 41.98185, -87.88133 41.9...",NaN,NaN
29786118,4686227143,0,320340442,tertiary,3,Bessie Coleman Drive,True,NaN,False,95.216379,"LINESTRING (-87.88573 41.97871, -87.88573 41.9...",30 mph,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
29785981,5037315526,0,31296557,unclassified,2,O'Hare International Terminal Departures,True,NaN,False,8.552609,"LINESTRING (-87.89083 41.97663, -87.89092 41.9...",20 mph,NaN
1153867719,102857949,0,11537330,motorway_link,NaN,NaN,True,NaN,False,15.968346,"LINESTRING (-87.87667 41.97976, -87.87658 41.9...",NaN,NaN
1153867766,102855117,0,19012965,motorway_link,1,NaN,True,NaN,False,22.372636,"LINESTRING (-87.88105 41.98147, -87.88105 41.9...",NaN,NaN


In [84]:
lf = pl.scan_parquet(DATASET_PATH)
df = lf.collect()
df.to_pandas()

,vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat
0,0,"[-87.8856713296918, 41.99494072305388]",0.5,1188647984_0,1188647984_0,1188647984,False,1188647984,-87.885671,41.994941
1,0,"[-87.88567733180858, 41.994936271618215]",1.0,1188647984_0,1188647984_0,1188647984,False,1188647984,-87.885677,41.994936
2,0,"[-87.88568911243644, 41.994927534580114]",1.5,1188647984_0,1188647984_0,1188647984,False,1188647984,-87.885689,41.994928
3,0,"[-87.88570568973026, 41.99491498626672]",2.0,:263785883_1_0,:263785883_1_0,321574768,False,node_263785883,-87.885706,41.994915
4,0,"[-87.88572492532072, 41.994900100759565]",2.5,:263785883_1_0,:263785883_1_0,321574768,False,node_263785883,-87.885725,41.994900
...,...,...,...,...,...,...,...,...,...,...
859435,1702,"[-87.88874921463292, 41.99804074104869]",4081.5,:10021303714_0_0,:10021303714_0_0,1094214001,False,node_10021303714,-87.888749,41.998041
859436,1702,"[-87.88875558331904, 41.99798545992872]",4082.0,1094214001_0,1094214001_0,1094214001,False,1094214001,-87.888756,41.997985
859437,1702,"[-87.88875619867935, 41.997933337316766]",4082.5,1094214001_0,1094214001_0,1094214001,False,1094214001,-87.888756,41.997933
859438,1702,"[-87.88875677920512, 41.99788060455132]",4083.0,1094214001_0,1094214001_0,1094214001,False,1094214001,-87.888757,41.997881


In [85]:
df.describe()

statistic,vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat
str,f64,f64,f64,str,str,str,f64,str,f64,f64
"""count""",859440.0,859440.0,859440.0,"""859440""","""859440""","""859035""",859440.0,"""859440""",859440.0,859440.0
"""null_count""",0.0,0.0,0.0,"""0""","""0""","""405""",0.0,"""0""",0.0,0.0
"""mean""",879.533613,null,1972.413852,null,null,null,0.058079,null,-87.889104,41.984193
"""std""",507.296619,null,1029.852968,null,null,null,null,null,0.007366,0.006624
"""min""",0.0,null,0.5,null,null,null,0.0,null,-87.906272,41.973193
"""25%""",443.0,null,1092.5,null,null,null,null,null,-87.892889,41.978784
"""50%""",872.0,null,1965.5,null,null,null,null,null,-87.885743,41.982073
"""75%""",1324.0,null,2851.5,null,null,null,null,null,-87.885582,41.98961
"""max""",1758.0,null,4083.5,null,null,null,1.0,null,-87.876231,41.999815


In [86]:
result = edges_gdf.reset_index().set_index("osmid").loc[321574770.0][["u", "v"]].values.tolist()
result

[np.int64(7710696745), np.int64(5493833658)]

In [87]:
from functools import lru_cache


@lru_cache(maxsize=None)
def ensure_connection(
    G_road: nx.MultiDiGraph, source: int, target: int
) -> set[int] | None:
    if source not in G_road or target not in G_road:
        return None
    try:
        path = nx.shortest_path(G_road, source=source, target=target, weight="length")
        edges_path = list((x, y, 0) for x, y in zip(path[:-1], path[1:]))
        edge_data = set(G.edges[edge]["osmid"] for edge in edges_path)
        return edge_data

    except nx.NetworkXNoPath:
        return None

In [88]:
df.filter(pl.col("node_mapped_id").cat.starts_with("node_"))

vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat
i64,"array[f64, 2]",f64,cat,cat,cat,bool,cat,f64,f64
0,"[-87.885706, 41.994915]",2.0,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885706,41.994915
0,"[-87.885725, 41.9949]",2.5,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885725,41.9949
0,"[-87.885748, 41.994882]",3.0,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885748,41.994882
0,"[-87.885771, 41.994857]",3.5,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885771,41.994857
0,"[-87.885789, 41.994827]",4.0,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885789,41.994827
…,…,…,…,…,…,…,…,…,…
1702,"[-87.888708, 41.999024]",4070.0,""":1350812521_6_0""",""":1350812521_6_0""","""1094498007""",false,"""node_1350812521""",-87.888708,41.999024
1702,"[-87.888756, 41.998723]",4075.0,""":10024343559_0_0""",""":10024343559_0_0""","""1094214000""",false,"""node_10024343559""",-87.888756,41.998723
1702,"[-87.888723, 41.998676]",4075.5,""":10024343559_0_0""",""":10024343559_0_0""","""1094214000""",false,"""node_10024343559""",-87.888723,41.998676


In [89]:
from typing import cast

import pandas as pd


indexed_osm_edges = edges_gdf.reset_index().set_index("osmid")

node_df = df.filter(pl.col("node_mapped_id").cat.starts_with("node_"))

# pairs: set[tuple[int, int]] = set()

# for row in node_df.iter_rows(named=True):
#     next_id = row.get("edge_id")
#     if next_id is None:
#         continue
#     next_id = int(next_id)

#     node_mapped_id = row.get("node_mapped_id")
#     if node_mapped_id is None:
#         continue

#     node_mapped_id = row.get("node_mapped_id")
#     if node_mapped_id is None:
#         print(f"Skipping row with missing node_mapped_id: {row}")
#         continue

#     # node_mapped_id can be like "node_1234" or already an int/np.int64
#     if isinstance(node_mapped_id, str) and node_mapped_id.startswith("node_"):
#         node_mapped_id = int(node_mapped_id.replace("node_", ""))
#     else:
#         node_mapped_id = int(node_mapped_id)
    
    

dfs_to_add = []
for row in node_df.iter_rows(named=True):
    # skip rows without an edge_id
    next_id = row.get("edge_id")
    if next_id is None:
        # edge_id is unmapped for this row; skip it
        # optionally you can log or collect these rows for inspection
        print(f"Skipping row with missing edge_id: {row["raw_lane_id"]}")
        continue

    # ensure next_id is an int (handles numpy int types as well)
    next_id = int(next_id)

    node_mapped_id = row.get("node_mapped_id")
    if node_mapped_id is None:
        print(f"Skipping row with missing node_mapped_id: {row}")
        continue

    # node_mapped_id can be like "node_1234" or already an int/np.int64
    if isinstance(node_mapped_id, str) and node_mapped_id.startswith("node_"):
        node_mapped_id = int(node_mapped_id.replace("node_", ""))
    else:
        node_mapped_id = int(node_mapped_id)

    try:
        sel = indexed_osm_edges.loc[next_id, ["u", "v"]] #type: ignore
        if isinstance(sel, (pd.Series,)):
            sel_df = sel.to_frame().T
        else:
            sel_df = sel
        edge = sel_df.values.tolist()
        if len(edge) == 0:
            print(f"No edge found for node {next_id}")
            continue
        for u, v in edge:
            if u != node_mapped_id:
                connect_edges = ensure_connection(
                    G, node_mapped_id, u
                ) 
                if connect_edges is not None:
                    # insert new edges into df
                    new_row = row.copy()
                    del new_row["edge_id"]
                    del new_row["time"]

                    new_df = pl.DataFrame(
                        [
                            {**new_row, "edge_id": str(connect_edge), "time": float(row["time"] + i * 0.1)}
                            for i, connect_edge in enumerate(connect_edges)
                        ]
                    )
                    new_df = new_df.with_columns(
                        pl.col("time").cast(pl.Float64),
                        pl.col("raw_lane_id").cast(pl.Categorical),
                        pl.col("vehicle_id").cast(pl.Int64),
                        pl.col("geo_position").cast(pl.Array(pl.Float64, shape=2)),
                        pl.col("edge_id").cast(pl.Categorical),
                        pl.col("node_mapped_id").cast(pl.Categorical),
                        pl.col("mapped_lane_id").cast(pl.Categorical),
                    )
                    dfs_to_add.append(new_df)
                    df = df.remove(pl.col("raw_lane_id") == row["raw_lane_id"])

    except KeyError:
        print(f"KeyError: {node_mapped_id}")


Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :310333211_0_0
Skipping row with missing edge_id: :310333211_0_0
Skipping row with missing edge_id: :310333211_0_0
Skipping row with missing edge_id: :6378331511_0_0
Skipping row with missing edge_id: :311482791_1_0
Skipping row with missing edge_id: :311482791_1_0
Skipping row with missing edge_id: :311482791_1_0
Skipping row with missing edge_id: :311482827_1_0
Skipping row with missing edge_id: :311482827_1_0
Skipping row with missing edge_id: :311482827_1_0
Skipping row with missing edge_id: :11038570643_0_0
Skipping row with missing edge_id: :11038570643_0_0
Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :12793948413_0_0
Skipping row with missing edge_id: :12793878899_2_0
Skipping row with missing edge_

In [90]:
df_to_add = pl.DataFrame(
        {
            "time": pl.Int64,
            "vehicle_id": pl.Int64,
            "geo_position": pl.Array(pl.Float64, shape=2),
            "edge_id": pl.Categorical,
            "mapped_lane_id": pl.Categorical,
            "node_mapped_id": pl.Categorical,
            "raw_lane_id": pl.Categorical,
        }
    )

if dfs_to_add:
    df_to_add = pl.concat(dfs_to_add)
    df_to_add = df_to_add.with_columns(
        pl.col("time").cast(pl.Float64),
        pl.col("vehicle_id").cast(pl.Int64),
        pl.col("geo_position").cast(pl.Array(pl.Float64, shape=2)),
        pl.col("edge_id").cast(pl.Categorical),
        pl.col("mapped_lane_id").cast(pl.Categorical),
        pl.col("node_mapped_id").cast(pl.Categorical),
        pl.col("raw_lane_id").cast(pl.Categorical),
    )

In [91]:
df_to_add.describe()

statistic,vehicle_id,geo_position,raw_lane_id,mapped_lane_id,reversed,node_mapped_id,lon,lat,edge_id,time
str,f64,f64,str,str,f64,str,f64,f64,str,f64
"""count""",941737.0,941737.0,"""941737""","""941737""",941737.0,"""941737""",941737.0,941737.0,"""941737""",941737.0
"""null_count""",0.0,0.0,"""0""","""0""",0.0,"""0""",0.0,0.0,"""0""",0.0
"""mean""",911.863554,null,null,null,0.0,null,-87.89635,41.98111,null,2022.488087
"""std""",499.542389,null,null,null,null,null,0.008355,0.004778,null,1014.002519
"""min""",0.0,null,null,null,0.0,null,-87.906268,41.974301,null,8.0
"""25%""",490.0,null,null,null,null,null,-87.903913,41.97727,null,1153.5
"""50%""",915.0,null,null,null,null,null,-87.899114,41.980161,null,2044.0
"""75%""",1364.0,null,null,null,null,null,-87.885731,41.984757,null,2922.0
"""max""",1758.0,null,null,null,0.0,null,-87.87807,41.999125,null,4081.5


In [92]:
df_to_add = df_to_add.select(df.columns)

concated_df = pl.concat([df, df_to_add], how="vertical")
concated_df = concated_df.sort(["time", "vehicle_id"])
concated_df

vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat
i64,"array[f64, 2]",f64,cat,cat,cat,bool,cat,f64,f64
0,"[-87.885671, 41.994941]",0.5,"""1188647984_0""","""1188647984_0""","""1188647984""",false,"""1188647984""",-87.885671,41.994941
0,"[-87.885677, 41.994936]",1.0,"""1188647984_0""","""1188647984_0""","""1188647984""",false,"""1188647984""",-87.885677,41.994936
0,"[-87.885689, 41.994928]",1.5,"""1188647984_0""","""1188647984_0""","""1188647984""",false,"""1188647984""",-87.885689,41.994928
0,"[-87.885706, 41.994915]",2.0,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885706,41.994915
0,"[-87.885725, 41.9949]",2.5,""":263785883_1_0""",""":263785883_1_0""","""321574768""",false,"""node_263785883""",-87.885725,41.9949
…,…,…,…,…,…,…,…,…,…
1702,"[-87.888749, 41.998041]",4081.5,""":10021303714_0_0""",""":10021303714_0_0""","""1094214001""",false,"""node_10021303714""",-87.888749,41.998041
1702,"[-87.888756, 41.997985]",4082.0,"""1094214001_0""","""1094214001_0""","""1094214001""",false,"""1094214001""",-87.888756,41.997985
1702,"[-87.888756, 41.997933]",4082.5,"""1094214001_0""","""1094214001_0""","""1094214001""",false,"""1094214001""",-87.888756,41.997933


In [93]:
concated_df.to_pandas()

,vehicle_id,geo_position,time,raw_lane_id,mapped_lane_id,edge_id,reversed,node_mapped_id,lon,lat
0,0,"[-87.8856713296918, 41.99494072305388]",0.5,1188647984_0,1188647984_0,1188647984,False,1188647984,-87.885671,41.994941
1,0,"[-87.88567733180858, 41.994936271618215]",1.0,1188647984_0,1188647984_0,1188647984,False,1188647984,-87.885677,41.994936
2,0,"[-87.88568911243644, 41.994927534580114]",1.5,1188647984_0,1188647984_0,1188647984,False,1188647984,-87.885689,41.994928
3,0,"[-87.88570568973026, 41.99491498626672]",2.0,:263785883_1_0,:263785883_1_0,321574768,False,node_263785883,-87.885706,41.994915
4,0,"[-87.88572492532072, 41.994900100759565]",2.5,:263785883_1_0,:263785883_1_0,321574768,False,node_263785883,-87.885725,41.994900
...,...,...,...,...,...,...,...,...,...,...
1740867,1702,"[-87.88874921463292, 41.99804074104869]",4081.5,:10021303714_0_0,:10021303714_0_0,1094214001,False,node_10021303714,-87.888749,41.998041
1740868,1702,"[-87.88875558331904, 41.99798545992872]",4082.0,1094214001_0,1094214001_0,1094214001,False,1094214001,-87.888756,41.997985
1740869,1702,"[-87.88875619867935, 41.997933337316766]",4082.5,1094214001_0,1094214001_0,1094214001,False,1094214001,-87.888756,41.997933
1740870,1702,"[-87.88875677920512, 41.99788060455132]",4083.0,1094214001_0,1094214001_0,1094214001,False,1094214001,-87.888757,41.997881


In [94]:
concated_df.write_parquet(DATASET_PATH.with_name("fcd_resolved.parquet"), compression="zstd")